In [9]:
import os
import numpy as np
import open3d as o3d
from sklearn.neighbors import NearestNeighbors
from scipy.spatial import KDTree
import warnings


In [ ]:
def load_point_cloud(file_path):
    """Загружает PLY с полями x,y,z,scalar_Label"""
    pcd = o3d.io.read_point_cloud(file_path, format='ply')
    data = np.loadtxt(file_path, skiprows=10)
    from plyfile import PlyData
    plydata = PlyData.read(file_path)
    x = plydata.elements[0].data['x']
    y = plydata.elements[0].data['y']
    z = plydata.elements[0].data['z']
    labels = plydata.elements[0].data['scalar_Label']
    points = np.vstack([x, y, z]).T
    return points, labels

def preprocess(points, labels, nb_neighbors=20, std_ratio=2.0):
    """Удаление шумов, нормализация, опциональный downsampling"""
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)

    pcd, ind = pcd.remove_statistical_outlier(nb_neighbors=nb_neighbors, std_ratio=std_ratio)
    points_clean = np.asarray(pcd.points)
    labels_clean = labels[ind]

    centroid = np.mean(points_clean, axis=0)
    points_clean = points_clean - centroid
    scale = np.max(np.linalg.norm(points_clean, axis=1))
    points_clean = points_clean / scale
    
    return points_clean, labels_clean


In [19]:
def get_segments(points, labels, min_points=50):
    """Возвращает словарь {label: points} без мелких сегментов"""
    unique_labels = np.unique(labels)
    segments = {}
    for lbl in unique_labels:
        mask = labels == lbl
        pts = points[mask]
        if len(pts) >= min_points:
            segments[lbl] = pts
    return segments

def compute_features(pts):
    """Вычисляет признаки для одного сегмента"""
    # PCA
    cov = np.cov(pts.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.sort(eigvals)[::-1]  # λ1 ≥ λ2 ≥ λ3
    total = np.sum(eigvals)
    linearity = (eigvals[0] - eigvals[1]) / (eigvals[0] + 1e-6)
    planarity = (eigvals[1] - eigvals[2]) / (eigvals[0] + 1e-6)
    sphericity = eigvals[2] / (eigvals[0] + 1e-6)

    if len(pts) > 10:
        nbrs = NearestNeighbors(n_neighbors=5).fit(pts)
        distances, _ = nbrs.kneighbors(pts)
        density = np.mean(distances[:, 1:])  # исключая себя
    else:
        density = 0.0
    
    normal_consistency = eigvals[1] / (eigvals[2] + 1e-6)
    
    if len(pts) > 10:
        from scipy.spatial import KDTree
        tree = KDTree(pts)
        edges = tree.query_pairs(r=2*density)
        if edges:
            import networkx as nx
            G = nx.Graph()
            G.add_edges_from(edges)
            components = nx.number_connected_components(G)
        else:
            components = len(pts)
    else:
        components = 1
    return {
        'linearity': linearity,
        'planarity': planarity,
        'sphericity': sphericity,
        'density': density,
        'normal_consistency': normal_consistency,
        'components': components,
        'eigvals': eigvals
    }

def classify_segment(features):
    """Возвращает тип: plane, tube, sphere, complex"""
    if features['planarity'] > 0.6:
        return 'plane'
    elif features['linearity'] > 0.6 and features['eigvals'][1] / (features['eigvals'][2]+1e-6) > 3.0:
        return 'tube'
    elif features['sphericity'] > 0.7:
        return 'sphere'
    else:
        return 'complex'

def select_method(seg_type, density):
    """Возвращает (method_name, params)"""
    if seg_type == 'plane':
        return 'alpha_shape', {'alpha': density * 2.0}
    elif seg_type == 'tube':
        return 'ball_pivoting', {'radii': [density * 1.5]}
    elif seg_type == 'sphere':
        return 'poisson', {'depth': 8}
    else:  # complex
        return 'poisson', {'depth': 10}

def reconstruct_segment(points, method, params):
    """Строит mesh для одного сегмента"""
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    mesh = None
    try:
        if method == 'poisson':
            mesh, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=params['depth'])
        elif method == 'alpha_shape':
            mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, params['alpha'])
        elif method == 'ball_pivoting':
            radii = params['radii']
            mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(pcd, o3d.utility.DoubleVector(radii))
    except Exception as e:
        warnings.warn(f"Reconstruction failed: {e}")
        mesh = None
    return mesh

def merge_meshes(meshes):
    """Объединяет список mesh в одну"""
    combined = o3d.geometry.TriangleMesh()
    for m in meshes:
        if m is not None and len(m.vertices) > 0:
            combined += m
    combined = combined.merge_close_vertices(1e-6)
    combined = combined.remove_degenerate_triangles()
    combined = combined.remove_unreferenced_vertices()
    return combined

def evaluate_reconstruction(original_points, mesh, sample_density=0.05):
    """RMSE от исходных точек до поверхности mesh"""
    if mesh is None or len(mesh.vertices) == 0:
        return float('inf')
    pcd_sampled = mesh.sample_points_uniformly(number_of_points=int(len(original_points)*0.5))
    sampled_pts = np.asarray(pcd_sampled.points)
    tree = KDTree(sampled_pts)
    distances, _ = tree.query(original_points)
    rmse = np.sqrt(np.mean(distances**2))
    return rmse

In [20]:
def process_one_file(file_path, visualize=False):
    points, labels = load_point_cloud(file_path)
    points_clean, labels_clean = preprocess(points, labels)
    segments = get_segments(points_clean, labels_clean, min_points=50)
    meshes = []
    metrics = {}
    for lbl, pts in segments.items():
        feat = compute_features(pts)
        seg_type = classify_segment(feat)
        method, params = select_method(seg_type, feat['density'])
        mesh = reconstruct_segment(pts, method, params)
        if mesh is not None:
            meshes.append(mesh)
        rmse = evaluate_reconstruction(pts, mesh)
        metrics[lbl] = {'type': seg_type, 'rmse': rmse}
    final_mesh = merge_meshes(meshes)
    if visualize and final_mesh.has_vertices():
        o3d.visualization.draw_geometries([final_mesh])
    return final_mesh, metrics

def process_folder(folder_path, output_dir):
    files = [f for f in os.listdir(folder_path) if f.endswith('.ply')]
    all_metrics = []
    for i, fname in enumerate(files[:500]):
        print(f"Processing {i+1}/500: {fname}")
        file_path = os.path.join(folder_path, fname)
        mesh, metrics = process_one_file(file_path)
        if mesh and mesh.has_vertices():
            out_mesh_path = os.path.join(output_dir, fname.replace('.ply', '_reconstructed.ply'))
            o3d.io.write_triangle_mesh(out_mesh_path, mesh)
        all_metrics.append(metrics)

    with open(os.path.join(output_dir, 'report.txt'), 'w') as f:
        f.write("===========================\n")
        for idx, m in enumerate(all_metrics):
            f.write(f"File {files[idx]}:\n")
            for lbl, info in m.items():
                f.write(f"  Segment {lbl}: type={info['type']}, RMSE={info['rmse']:.5f}\n")
    print("Finish")

In [23]:
# Пример запуска:
folder = "dataset"
output = "results"
process_folder(folder, output)

Processing 1/500: valve_0001_lidar_classes.ply
Processing 2/500: valve_0002_lidar_classes.ply
Processing 3/500: valve_0003_lidar_classes.ply
Processing 4/500: valve_0004_lidar_classes.ply
Processing 5/500: valve_0005_lidar_classes.ply
Processing 6/500: valve_0006_lidar_classes.ply
Processing 7/500: valve_0007_lidar_classes.ply
Processing 8/500: valve_0008_lidar_classes.ply
Processing 9/500: valve_0009_lidar_classes.ply
Processing 10/500: valve_0010_lidar_classes.ply
Processing 11/500: valve_0011_lidar_classes.ply
Processing 12/500: valve_0012_lidar_classes.ply
Processing 13/500: valve_0013_lidar_classes.ply
Processing 14/500: valve_0014_lidar_classes.ply
Processing 15/500: valve_0015_lidar_classes.ply
Processing 16/500: valve_0016_lidar_classes.ply
Processing 17/500: valve_0017_lidar_classes.ply
Processing 18/500: valve_0018_lidar_classes.ply
Processing 19/500: valve_0019_lidar_classes.ply
Processing 20/500: valve_0020_lidar_classes.ply
Processing 21/500: valve_0021_lidar_classes.ply
P